<a href="https://colab.research.google.com/github/TOTVScontext/DataScience/blob/main/Context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python -m spacy download pt_core_news_sm
!pip install transformers

In [ ]:
df['TEXTO_LIMPO'] = df['ANON_TRANSCRICAO'].apply(limpar_transcricao)

In [ ]:
print(df.columns)


In [ ]:
import pandas as pd
import re
import nltk
from collections import Counter
import plotly.express as px
import spacy
from IPython.display import display

nlp = spacy.load("pt_core_news_sm")
nltk.download('stopwords')
stop_words_pt = set(nltk.corpus.stopwords.words('portuguese'))

stop_words_custom = {
    'pra', 'aí', 'então', 'né', 'tá', 'gente', 'aqui', 'vai', 'fazer',
    'ter', 'porque', 'lá', 'assim', 'ser', 'vou', 'agora', 'tudo',
    'bem', 'entendi', 'falou', 'acho', 'vamos', 'pode', 'sobre', 'dá',
    'voc', 'você', 'n', 't', 'pro', 'mim', 'coisa', 'parte', 'dia', 'exemplo',
    'cara'
}

#Baseado noq foi encontrado com n-grams
categorias = {
    'Financeiro': 'contas pagar|pagar receber|financeiro|boleto|faturamento|nota|fical|pix|cartão|transferência|valor',
    'RH/Operacional': 'relógio ponto|ponto|jornada|colaborador|horas',
    'Vendas/Produtividade': 'vendedor|otimizar|venda|proposta|pipeline|crm',
    'Atendimento': 'falar|cliente|atendimento|suporte|contato'
}
stop_words_pt.update(stop_words_custom)

df = pd.read_csv('ANON_nome_transcricao.csv', sep=',', on_bad_lines='skip', encoding='utf-8', encoding_errors='ignore')


#LIMPEZA E EXTRAÇÃO

def limpar_transcricao(texto):
    if pd.isna(texto):
        return ""
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúãõçâêôà\s]', ' ', texto)
    texto = ' '.join(texto.split())
    return texto

def extrair_palavras_chave(texto):
    palavras = texto.split()
    palavras_limpas = [p for p in palavras if p not in stop_words_pt and len(p) > 2]
    return palavras_limpas

def extrair_palavras_chave_spacy(texto):
    doc = nlp(texto)
    palavras_limpas = [
        token.text for token in doc
        if token.text not in stop_words_pt
        and len(token.text) > 2
        and token.pos_ == "NOUN"
    ]
    return palavras_limpas

def categorizar_reuniao(texto):
  tags_encontradas = []
  for tema, palavras_chave in categorias.items():
    if re.search(palavras_chave, texto, re.IGNORECASE):
      tags_encontradas.append(tema)
  return tags_encontradas if tags_encontradas else ['Geral']


#--------------------------------------------------------------------------------------------------------------
#Ia para achar os freagmentos das reunioes sobre preco e incertezas
from transformers import pipeline
#tecnica de zero-shot
classificador = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

if 'TEXTO_LIMPO' not in df.columns:
    print("A coluna estava faltando! Criando TEXTO_LIMPO agora...")
    df['TEXTO_LIMPO'] = df['ANON_TRANSCRICAO'].apply(limpar_transcricao)
df_amostra = df.head(1000).copy()

def identificar_objecoes_ia(texto):
    if len(texto) < 10: return "Neutro"
    labels_objetivo = ["problema de orçamento ou preço", "dificuldade técnica ou dúvida", "menção a empresa concorrente", "satisfação e elogio"]
    resultado = classificador(texto, labels_objetivo, hypothesis_template="Este trecho indica {}.")

    top_label = resultado['labels'][0]
    top_score = resultado['scores'][0]
    return top_label if top_score > 0.4 else "Assunto Geral"

df_real = df.dropna(subset=['ANON_TRANSCRICAO']).copy()
amostra_viva = df_real.sample(5)

print("Agora sim! Analisando textos que existem...")
amostra_viva['IA_OBJECOES'] = amostra_viva['TEXTO_LIMPO'].apply(identificar_objecoes_ia)

display(amostra_viva[['TEXTO_LIMPO', 'IA_OBJECOES']])
#----------------------------------------------------------------------------------------------------


#Texto limpo
print("Aplicando limpeza pesada e processando vocabulário...")
df['TEXTO_LIMPO'] = df['ANON_TRANSCRICAO'].apply(limpar_transcricao)
#logica de gatilhos (proximos possiveis passos)
gatilhos = 'whatsapp|ligar|contato|agendar|retorno|proxima|numero|reuniao'
df['TEM_PROXIMO_PASSO'] = df['TEXTO_LIMPO'].str.contains(gatilhos, case=False, na=False)

print("--- CONTADOR DE GATILHOS ---")
print(df['TEM_PROXIMO_PASSO'].value_counts())

# melhoria do codigo da ia de verificacao
df_ouro = df[(df['TEM_PROXIMO_PASSO'] == True) & (df['TEXTO_LIMPO'].str.len() > 20)].copy()
print(f"Filtragem concluída! Das 87 reuniões, temos {len(df_ouro)} prontas para a análise profunda.")
print("Iniciando a IA nas reuniões de elite... aguarde.")
df_ouro['IA_OBJECOES'] = df_ouro['TEXTO_LIMPO'].apply(identificar_objecoes_ia)
print("\n--- RESULTADO DO TESTE DE OURO ---")
# Mostramos o texto e o que a IA interpretou dele
display(df_ouro[['TEXTO_LIMPO', 'IA_OBJECOES']].head(15))
#

print("\n--- AMOSTRA DOS 10 PRIMEIROS GATILHOS ENCONTRADOS ---")
display(df[df['TEM_PROXIMO_PASSO'] == True][['TEXTO_LIMPO']].head(10))

#TExto limpo so de substantivos essenciais
df['TOKENS'] = df['TEXTO_LIMPO'].apply(extrair_palavras_chave_spacy)

#TAGS
df['TAGS'] = df['TEXTO_LIMPO'].apply(categorizar_reuniao)
df_tags_contagem = df.explode('TAGS')
df_interessante = df_tags_contagem[df_tags_contagem['TAGS'] != 'Geral']
fig_temas = px.bar(df_interessante['TAGS'].value_counts().reset_index(),
                   x='count', y='TAGS', orientation='h',
                   title='Pilar 1 (B): Distribuição de Temas (Tags)',
                   color='count', color_continuous_scale='Viridis')
fig_temas.show()


# filtro final pilar 1
import plotly.express as px

df_final_viz = df_ouro[~df_ouro['IA_OBJECOES'].isin(['Assunto Geral', 'Neutro'])]

fig_objecoes = px.pie(df_final_viz, names='IA_OBJECOES',
             title='Pilar 1 (C): Análise Semântica de Objeções (IA Deep Learning)',
             hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Pastel)

fig_objecoes.update_traces(textinfo='percent+label')
fig_objecoes.show()

# 3. Um resumo rápido no console para conferir
print(f"Total de reuniões analisadas pela IA: {len(df_ouro)}")
print(f"Objeções claras detectadas: {len(df_final_viz)}")
#TERMOS MAIS FREQUENTES

todas_palavras = [palavra for sublist in df['TOKENS'] for palavra in sublist]
contagem = Counter(todas_palavras)

top_palavras = pd.DataFrame(contagem.most_common(20), columns=['Palavra', 'Frequência'])

fig_palavras = px.bar(top_palavras, x='Frequência', y='Palavra',
                      orientation='h',
                      title='Top 20 Termos Reais de Negócios (Dados Limpos)',
                      color='Frequência',
                      color_continuous_scale=px.colors.sequential.Teal)
fig_palavras.update_layout(yaxis={'categoryorder':'total ascending'})
fig_palavras.show()



#Codigo gerado por ia para achar padroes e identificar as categorias principais das reunioes (UTILIZADO SOMENTE PARA FINS DE PRODUTIVIDADE)
'''from sklearn.feature_extraction.text import CountVectorizer

docs = df['TEXTO_LIMPO'].dropna().head(10000)
vec = CountVectorizer(ngram_range=(2, 2), stop_words=list(stop_words_pt)).fit(docs)
bag_of_words = vec.transform(docs)
sum_words = bag_of_words.sum(axis=0)

words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)

print("Os 15 assuntos (pares) mais falados são:")
display(words_freq[:15])'''



In [ ]:
# Vamos ver o que tem nas 10 primeiras linhas de 'TEXTO_LIMPO' e da original
display(df[['ANON_TRANSCRICAO', 'TEXTO_LIMPO']].head(10))